# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore a FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following best practices with Croissant schemas and meticulous referencing by `@id` fields.

### Dataset Source
The dataset is represented by a Croissant schema available here: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Install mlcroissant if not present
!pip install mlcroissant

## 1. Data Loading
We load dataset metadata and records using `mlcroissant.Dataset`. This provides access to all entities, their `@id`s, and the data contents.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Let's examine the available record sets and their associated field and column `@id`s. ALL entities are referenced by their Croissant `@id` identifiers.

Below, we print all available record set `@id`s together with their field and column `@id`s.

In [ ]:
# List record set @ids, fields, and columns
record_sets = list(dataset.record_sets)
print("Available record sets and their fields/columns:")
for rs in record_sets:
    print(f'  RecordSet @id: {rs.id}')
    all_field_ids = [field.id for field in rs.fields]
    all_column_ids = []
    for field in rs.fields:
        if hasattr(field, 'columns'):
            for col in getattr(field, 'columns', []):
                # In Croissant, both fields and columns may have @id
                if hasattr(col, 'id'):
                    all_column_ids.append(col.id)
    print(f'    Fields: {all_field_ids}')
    if all_column_ids:
        print(f'    Columns: {all_column_ids}')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Only `@id` values are used for reference.

Let's load the first (or main) record set. You can adapt the code to load other record sets as needed.

In [ ]:
# Gather DataFrames from each record set, accessible via @id
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df

# Choose the first record set as main
main_rs_id = record_sets[0].id if record_sets else None
if main_rs_id:
    print(f"Record set @id: {main_rs_id}")
    print("Columns (field @id, or as named in schema):", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Let's pick a numeric field for further analysis. You can adjust the field by using the correct `@id` (or schema-specified name) from the record set printed above.

The typical numeric fields in such proteomics datasets are `cr:peptide_count` (for peptide counts), `cr:MW` (molecular weight), or `cr:coverage_percent` (sequence coverage), but consult your field list above to set `numeric_field_id` accordingly.

In [ ]:
# Set the numeric field for analysis (replace by exact @id or column name printed above if needed)
numeric_field_id = None
if main_rs_id:
    # Pick a standard plausible field name for demo; adjust field id to match your schema
    for c in dataframes[main_rs_id].columns:
        if c.lower() in ['cr:peptide_count', 'peptide_count', 'cr:mw', 'mw', 'cr:coverage_percent', 'coverage']:
            numeric_field_id = c
            break
    if numeric_field_id is None:
        # Fallback to first numeric column
        for c in dataframes[main_rs_id].columns:
            if pd.api.types.is_numeric_dtype(dataframes[main_rs_id][c]):
                numeric_field_id = c
                break
    print(f"Selected numeric field: {numeric_field_id}")

    # Filtering and normalization
    threshold = 10
    if numeric_field_id:
        filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Attempt to group by a categorical field (e.g., sample or accession)
        group_field = None
        for c in dataframes[main_rs_id].columns:
            if c.lower() in ['cr:accession', 'accession', 'cr:sample', 'sample']:
                group_field = c
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (means):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No record set loaded - cannot perform EDA.")

## 5. Visualization
Let's plot the distribution of the selected numeric field and visualize its relationship to another feature if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_rs_id][numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Optionally scatter against another numeric variable
    numeric_columns = [c for c in dataframes[main_rs_id].columns if pd.api.types.is_numeric_dtype(dataframes[main_rs_id][c]) and c != numeric_field_id]
    if numeric_columns:
        plt.figure(figsize=(6, 6))
        sns.scatterplot(
            x=dataframes[main_rs_id][numeric_field_id],
            y=dataframes[main_rs_id][numeric_columns[0]]
        )
        plt.xlabel(numeric_field_id)
        plt.ylabel(numeric_columns[0])
        plt.title(f"Scatter: {numeric_field_id} vs. {numeric_columns[0]}")
        plt.show()
else:
    print("Nothing to plot - missing data or unknown numeric field.")

## 6. Conclusion
This notebook illustrated loading and analyzing a complex dataset defined in Croissant, using strict reference by `@id`. You can extend this workflow to more complex queries or apply machine learning tools directly on the extracted DataFrames.